In [14]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

path='stores_sales_forecasting.csv'
df = pd.read_csv('stores_sales_forecasting.csv', encoding='latin1')
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [15]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

os.makedirs('output', exist_ok=True)
df=pd.read_csv('stores_sales_forecasting.csv', encoding='latin1')
df['Order Date']=pd.to_datetime(df['Order Date'], errors='coerce')
df=df.dropna(subset=['Order Date','Sales']).sort_values('Order Date')
monthly=df.set_index('Order Date').resample('MS')['Sales'].sum().reset_index()
monthly['month_num']=np.arange(len(monthly))
monthly['lag1']=monthly['Sales'].shift(1)
monthly['lag2']=monthly['Sales'].shift(2)
monthly['rolling3']=monthly['Sales'].shift(1).rolling(3).mean()
monthly=monthly.dropna().reset_index(drop=True)
train=monthly.iloc[:-6].copy(); test=monthly.iloc[-6:].copy()
X_train=train[['month_num','lag1','lag2','rolling3']]; y_train=train['Sales']
X_test=test[['month_num','lag1','lag2','rolling3']]; y_test=test['Sales']
model=RandomForestRegressor(n_estimators=500, random_state=42)
model.fit(X_train, y_train)
pred=model.predict(X_test)
metrics = {
    'MAE': mean_absolute_error(y_test, pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
    'R2': r2_score(y_test, pred)
}
future=[]
last=monthly.copy()
for i in range(6):
    mnum=last['month_num'].iloc[-1]+1
    lag1=last['Sales'].iloc[-1]
    lag2=last['Sales'].iloc[-2]
    rolling3=last['Sales'].iloc[-3:].mean()
    x=pd.DataFrame([[mnum,lag1,lag2,rolling3]], columns=['month_num','lag1','lag2','rolling3'])
    yhat=float(model.predict(x)[0])
    new_date=(last['Order Date'].iloc[-1]+pd.offsets.MonthBegin(1))
    future.append([new_date, yhat])
    last=pd.concat([last, pd.DataFrame({'Order Date':[new_date],'Sales':[yhat],'month_num':[mnum],'lag1':[lag1],'lag2':[lag2],'rolling3':[rolling3]})], ignore_index=True)
future_df=pd.DataFrame(future, columns=['Order Date','Forecast Sales'])
summary=pd.DataFrame({'Metric':list(metrics.keys()), 'Value':list(metrics.values())})
summary.to_csv('output/sales_forecasting_metrics.csv', index=False)
future_df.to_csv('output/sales_forecasting_output.csv', index=False)
print(summary)
print(future_df)

  Metric         Value
0    MAE   9757.776597
1   RMSE  12242.820058
2     R2     -0.890446
  Order Date  Forecast Sales
0 2018-01-01    14769.506479
1 2018-02-01    10861.702146
2 2018-03-01    18679.722173
3 2018-04-01    18281.204845
4 2018-05-01    18493.329956
5 2018-06-01    20095.037076


In [16]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

os.makedirs('output', exist_ok=True)
df=pd.read_csv('stores_sales_forecasting.csv', encoding='latin1')
df['Order Date']=pd.to_datetime(df['Order Date'], errors='coerce')
df=df.dropna(subset=['Order Date','Sales']).sort_values('Order Date')
monthly=df.set_index('Order Date').resample('MS')['Sales'].sum().reset_index()
monthly['month_num']=np.arange(len(monthly))
monthly['lag1']=monthly['Sales'].shift(1)
monthly['lag2']=monthly['Sales'].shift(2)
monthly['rolling3']=monthly['Sales'].shift(1).rolling(3).mean()
monthly=monthly.dropna().reset_index(drop=True)
train=monthly.iloc[:-6].copy(); test=monthly.iloc[-6:].copy()
X_train=train[['month_num','lag1','lag2','rolling3']]; y_train=train['Sales']
X_test=test[['month_num','lag1','lag2','rolling3']]; y_test=test['Sales']
model=RandomForestRegressor(n_estimators=500, random_state=42)
model.fit(X_train, y_train)
pred=model.predict(X_test)
metrics={'MAE':mean_absolute_error(y_test,pred),'RMSE':mean_squared_error(y_test,pred)**0.5,'R2':r2_score(y_test,pred)}
future=[]
last=monthly.copy()
for i in range(6):
    mnum=last['month_num'].iloc[-1]+1
    lag1=last['Sales'].iloc[-1]
    lag2=last['Sales'].iloc[-2]
    rolling3=last['Sales'].iloc[-3:].mean()
    x=pd.DataFrame([[mnum,lag1,lag2,rolling3]], columns=['month_num','lag1','lag2','rolling3'])
    yhat=float(model.predict(x)[0])
    new_date=(last['Order Date'].iloc[-1]+pd.offsets.MonthBegin(1))
    future.append([new_date, yhat])
    last=pd.concat([last, pd.DataFrame({'Order Date':[new_date],'Sales':[yhat],'month_num':[mnum],'lag1':[lag1],'lag2':[lag2],'rolling3':[rolling3]})], ignore_index=True)
future_df=pd.DataFrame(future, columns=['Order Date','Forecast Sales'])
summary=pd.DataFrame({'Metric':list(metrics.keys()), 'Value':list(metrics.values())})
summary.to_csv('output/sales_forecasting_metrics.csv', index=False)
future_df.to_csv('output/sales_forecasting_output.csv', index=False)
print(summary.to_string(index=False))
print(future_df.to_string(index=False))

Metric        Value
   MAE  9757.776597
  RMSE 12242.820058
    R2    -0.890446
Order Date  Forecast Sales
2018-01-01    14769.506479
2018-02-01    10861.702146
2018-03-01    18679.722173
2018-04-01    18281.204845
2018-05-01    18493.329956
2018-06-01    20095.037076


In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import os
os.makedirs('output', exist_ok=True)
metrics=pd.read_csv('output/sales_forecasting_metrics.csv')
forecast=pd.read_csv('output/sales_forecasting_output.csv')
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(forecast['Order Date'], forecast['Forecast Sales'], color='#4C78A8')
ax.set_title('Forecasted Monthly Sales')
ax.set_xlabel('Month')
ax.set_ylabel('Sales')
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig('output/sales_forecast_chart.png', dpi=200)
plt.close(fig)
print('done')

done
